In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
TRAIN_PATH = "./../data/processed/train2.csv"
VALIDATION_PATH = "./../data/processed/validation2.csv"
TEST_PATH = "./../data/processed/test2.csv"

In [3]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_validation = validation_df.drop(columns=["label"])
y_validation = validation_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

del train_df, validation_df, test_df

In [4]:
#keep turnid and convid for later use in error analysis and model explainability
X_train_ids = X_train[["turn_id", "conv_id"]]
X_validation_ids = X_validation[["turn_id", "conv_id"]]
X_test_ids = X_test[["turn_id", "conv_id"]]

X_train = X_train.drop(columns=["turn_id", "conv_id"])
X_validation = X_validation.drop(columns=["turn_id", "conv_id"])
X_test = X_test.drop(columns=["turn_id", "conv_id"])

In [5]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (2636, 359)
Validation set shape: (293, 359)
Test set shape: (326, 359)


In [6]:
def save_confusion_matrix(cm, filename,title):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.savefig(filename)

    plt.close()

In [7]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

In [8]:
from evaluate import MultiTurnJailbreakEvaluator
import pandas as pd

evaluator = MultiTurnJailbreakEvaluator()

def group_by_conv(X, y_true, y_pred):
 
    df = X.copy()
    df["_label"] = y_true
    df["_pred"]  = y_pred

    # sort so turns are in order within each conversation
    df = df.sort_values(["conv_id"]).reset_index(drop=True)

    conv_ids       = []
    y_true_grouped = []
    y_pred_grouped = []

    for cid, group in df.groupby("conv_id", sort=False):
        conv_ids.append(cid)
        y_true_grouped.append(group["_label"].tolist())
        y_pred_grouped.append(group["_pred"].tolist())

    return conv_ids, y_true_grouped, y_pred_grouped


In [9]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=min(10, len(X_train)-1),
    contamination=.2,
    metric="cosine"
)

y_pred_train = lof.fit_predict(X_train, y_train)
y_pred_train = [1 if p == -1 else 0 for p in y_pred_train]
acc_train, report_train, cm_train = evaluate_model(y_pred_train, y_train)



In [10]:
print ("LOF Train Accuracy:", acc_train)
print ("LOF Train Classification Report:\n", report_train)

LOF Train Accuracy: 0.6532625189681336
LOF Train Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.81      0.77      1923
           1       0.31      0.23      0.26       713

    accuracy                           0.65      2636
   macro avg       0.52      0.52      0.52      2636
weighted avg       0.62      0.65      0.64      2636



In [11]:

params = {
    'penalty': 'l2',         
    'C': 1.0,                
    'solver': 'lbfgs',       
    'max_iter': 10000,        
    'random_state': 42
}

logreg = LogisticRegression(**params)
logreg.fit(X_train, y_train)

y_train_pred = logreg.predict(X_train)
y_validation_pred = logreg.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)


test_acc, test_report, test_cm = evaluate_model(logreg.predict(X_test), y_test)


Training Accuracy: 0.8190440060698028
Validation Accuracy: 0.7679180887372014
Training Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88      1923
           1       0.73      0.53      0.61       713

    accuracy                           0.82      2636
   macro avg       0.78      0.73      0.75      2636
weighted avg       0.81      0.82      0.81      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       209
           1       0.62      0.49      0.55        84

    accuracy                           0.77       293
   macro avg       0.72      0.68      0.70       293
weighted avg       0.76      0.77      0.76       293



In [12]:
with open("../reports/evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:logistic_regression \n" \
    "==============================\n" \
    "param_grid: {'C': 1.0, 'random_state': 42}\n" )
    # f.write("cv_best_score: " + str(grid_search.best_score_) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_acc: " + str(test_acc) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    


In [13]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.8190440060698028
Validation Accuracy: 0.7679180887372014
Training Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88      1923
           1       0.73      0.53      0.61       713

    accuracy                           0.82      2636
   macro avg       0.78      0.73      0.75      2636
weighted avg       0.81      0.82      0.81      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       209
           1       0.62      0.49      0.55        84

    accuracy                           0.77       293
   macro avg       0.72      0.68      0.70       293
weighted avg       0.76      0.77      0.76       293

Test Accuracy: 0.7730061349693251
Test Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.85      0.84       226
           1       0.64     

In [14]:
params = {
    'max_depth': 5,           
    'min_samples_split': 5,    
    'min_samples_leaf': 10,     
    'max_features': None,      
    'random_state': 42
}

dtc = DecisionTreeClassifier(**params)
dtc.fit(X_train, y_train)

y_train_pred = dtc.predict(X_train)
y_validation_pred = dtc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


test_acc, test_report, test_cm = evaluate_model(dtc.predict(X_test), y_test)

In [15]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.840288315629742
Validation Accuracy: 0.78839590443686
Training Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.93      0.89      1923
           1       0.76      0.60      0.67       713

    accuracy                           0.84      2636
   macro avg       0.81      0.77      0.78      2636
weighted avg       0.83      0.84      0.83      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.92      0.86       209
           1       0.70      0.45      0.55        84

    accuracy                           0.79       293
   macro avg       0.76      0.69      0.71       293
weighted avg       0.78      0.79      0.77       293

Test Accuracy: 0.8282208588957055
Test Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.92      0.88       226
           1       0.78      0.

In [16]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:decision_tree \n" \
    "==============================\n" \
    "param_grid: {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2 , 'random_state': 42 ,''}\n" )

    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )


In [17]:
params = {
'n_estimators': 200,          
'max_depth': 5,             
'min_samples_split': 5,      
'min_samples_leaf': 5,        
'max_features': 'sqrt',       
'bootstrap': True,
'random_state': 42,
'n_jobs': -1                 
}

rfc = RandomForestClassifier(**params)
rfc.fit(X_train, y_train)
y_train_pred = rfc.predict(X_train)
y_validation_pred = rfc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


test_acc, test_report, test_cm = evaluate_model(rfc.predict(X_test), y_test)

d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [18]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)  
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)  

Training Accuracy: 0.7344461305007587
Validation Accuracy: 0.7133105802047781
Training Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.85      1923
           1       1.00      0.02      0.04       713

    accuracy                           0.73      2636
   macro avg       0.87      0.51      0.44      2636
weighted avg       0.81      0.73      0.63      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.71      1.00      0.83       209
           1       0.00      0.00      0.00        84

    accuracy                           0.71       293
   macro avg       0.36      0.50      0.42       293
weighted avg       0.51      0.71      0.59       293

Test Accuracy: 0.6932515337423313
Test Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       226
           1       0.00     

In [19]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:random_forest_tuned_v1 \n" \
    "==============================\n" \
    "param_grid: {'n_estimators': 300,'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2,  'max_features': 'sqrt','bootstrap': True,'random_state': 42,'n_jobs': -1}'\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )



In [20]:
params = {
    'n_estimators': 300,        
    'max_depth': 3,             
    'learning_rate': 0.01,       
    'random_state': 42,
    'n_jobs': -1,
}

xgb = XGBClassifier(**params)
xgb.fit(X_train, y_train)

# Predictions
y_train_pred = xgb.predict(X_train)
y_validation_pred = xgb.predict(X_validation)

# Evaluation
train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)
test_acc, test_report, test_cm = evaluate_model(xgb.predict(X_test), y_test)

In [21]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.8869499241274659
Validation Accuracy: 0.8225255972696246
Training Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.95      0.92      1923
           1       0.84      0.72      0.78       713

    accuracy                           0.89      2636
   macro avg       0.87      0.83      0.85      2636
weighted avg       0.88      0.89      0.88      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.88       209
           1       0.76      0.56      0.64        84

    accuracy                           0.82       293
   macro avg       0.80      0.74      0.76       293
weighted avg       0.82      0.82      0.81       293

Test Accuracy: 0.8404907975460123
Test Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.94      0.89       226
           1       0.82     

In [22]:
with open("evaluation_report.txt", "a") as f:
    # write data of the model 
    
    f.write("==============================\n" \
    "Model:xgboost \n" \
    "==============================\n" \
    "param_grid: " + str(params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
    

In [23]:
params = {
    'C': 1.0,                
    'kernel': 'rbf',        
    'random_state': 42
}

svc = SVC(**params)
svc.fit(X_train, y_train)

y_train_pred = svc.predict(X_train)
y_validation_pred = svc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

test_acc, test_report, test_cm = evaluate_model(svc.predict(X_test), y_test)



d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Apps\anaconda\envs\langgraph-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [24]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7306525037936267
Validation Accuracy: 0.7064846416382252
Training Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.84      1923
           1       1.00      0.00      0.01       713

    accuracy                           0.73      2636
   macro avg       0.87      0.50      0.43      2636
weighted avg       0.80      0.73      0.62      2636

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.99      0.83       209
           1       0.00      0.00      0.00        84

    accuracy                           0.71       293
   macro avg       0.36      0.50      0.41       293
weighted avg       0.51      0.71      0.59       293

Test Accuracy: 0.6932515337423313
Test Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       226
           1       0.00     

In [25]:
with open("evaluation_report.txt", "a") as f:
   
    f.write("==============================\n" \
    "Model: Support Vector Classifier\n" \
    "==============================\n" \
    "param_grid: " + str(params) + "\n" )
    f.write("train_acc: " + str(train_acc) + "\n" )
    f.write("val_acc: " + str(val_acc) + "\n" )
    f.write("train_report: " + str(train_report) + "\n" )
    f.write("val_report: " + str(val_report) + "\n" )
    f.write("test_report: " + str(test_report) + "\n" )
